In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -U ultralytics
!pip install -U torch torchvision torchaudio  # Ensure latest PyTorch for GPU support

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 27.9 MB/s eta 0:00:00


In [3]:
import torch
import ultralytics
ultralytics.checks()
print(f"Using GPU: {torch.cuda.is_available()}")

Ultralytics 8.3.200 🚀 Python-3.12.11 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 39.0/112.6 GB disk)
Using GPU: True


In [ ]:
import shutil
import os

project_dir = '/content/CFNet_Project'
os.makedirs(project_dir, exist_ok=True)

shutil.copytree('/content/drive/MyDrive/CFNet_Project/dataset', f'{project_dir}/dataset')
shutil.copy('/content/drive/MyDrive/CFNet_Project/cfnet.yaml', f'{project_dir}/cfnet.yaml')

In [ ]:
from ultralytics import YOLO

model = YOLO(f'{project_dir}/cfnet.yaml')

model = YOLO('yolov10n.pt')
model = model.load(f'{project_dir}/cfnet.yaml')

print(model)

In [ ]:
# Train the model
results = model.train(
    data=f'{project_dir}/dataset/data.yaml',  # Path to your data.yaml
    epochs=100,  # 50-200; start low and monitor
    imgsz=640,  # Input size; 416 for faster, 640 for accuracy
    batch=16,  # Adjust based on GPU (8-32; T4 handles 16)
    name='cfnet_cantaloupe',  # Experiment name (saves in /runs/detect/)
    device=0,  # GPU (0 for first GPU)
    workers=4,  # Data loaders
    optimizer='AdamW',  # Good for fine-tuning
    lr0=0.001,  # Initial learning rate
    patience=20,  # Early stopping if no val improvement
    augment=True,  # Enable augmentations
    mosaic=1.0,  # High for small objects like buds
    mixup=0.2,  # Blending for variety
    degrees=20.0,  # Rotation for plant angles
    translate=0.15,  # Shift for occlusions
    scale=0.6,  # Scaling
    shear=3.0,  # Perspective
    fliplr=0.5,  # Flip
    hsv_h=0.02,  # Hue for lighting
    hsv_s=0.8,  # Saturation
    hsv_v=0.5,  # Value
    # If class imbalance: cls_weights=[1.0, 1.2] (e.g., more weight to flowers if rarer)
)

shutil.copytree('/content/runs/detect/cfnet_cantaloupe', '/content/drive/MyDrive/CFNet_Project/runs/cfnet_cantaloupe')

In [ ]:
# Load best model
best_model = YOLO('/content/runs/detect/cfnet_cantaloupe/weights/best.pt')

# Validate
metrics = best_model.val()
print(metrics)  # mAP, precision, recall, etc.

# Confusion matrix and plots are saved in /runs/detect/val/
shutil.copytree('/content/runs/detect/val', '/content/drive/MyDrive/CFNet_Project/runs/val')

In [ ]:
results = best_model.predict(source='path/to/test_image.jpg', conf=0.5, iou=0.45)
results[0].show()  # Display with boxes

results = best_model.predict(source=0, stream=True)  # Webcam

In [ ]:
best_model.export(format='onnx')  # ONNX
best_model.export(format='engine')  # TensorRT

shutil.copy('/content/runs/detect/cfnet_cantaloupe/weights/best.onnx', '/content/drive/MyDrive/CFNet_Project/best.onnx')